# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) ?? ??nh v? h? h?i m?t ???ng
Ch?y l?n l??t c?c b??c 0?8 tr?n GPU T4. M?i th? nghi?m b?t ??u t? model g?c; k?t qu? ???c l?u trong `work/` v? `report/`.

## 0. Môi trường (Colab T4)

In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
# Lab uses bitsandbytes; remove optional torchao before importing PEFT.
%pip uninstall -y torchao
%pip -q install -r requirements-colab.txt

In [ ]:
import os
from pathlib import Path

# Kaggle d?ng dataset ?? g?n; Colab l?y th?ng tin ??ng nh?p t? Secrets.
if not Path('/kaggle/input').exists():
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

## 1. Dữ liệu: tải và chia split

In [ ]:
import subprocess
import sys
from pathlib import Path

prepare_script = Path('scripts/prepare_data.py')
if not prepare_script.is_file():
    prepare_script = Path('K4-Track-4-Day10-VisionLanguageModel/scripts/prepare_data.py')
command = [sys.executable, str(prepare_script.resolve())]
command += ['--source', '/kaggle/input'] if Path('/kaggle/input').exists() else ['--kaggle']
subprocess.run(command, cwd=str(prepare_script.resolve().parents[1]), check=True)

In [ ]:
import json
import random
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

# Chay duoc khi thu muc hien tai la thu muc lab hoac thu muc cha.
lab_dir = Path.cwd()
if not (lab_dir / "data/splits/train.jsonl").exists():
    lab_dir = lab_dir / "K4-Track-4-Day10-VisionLanguageModel"
lab_dir = lab_dir.resolve()
data_dir = lab_dir / "data"
if str(lab_dir) not in sys.path:
    sys.path.insert(0, str(lab_dir))

splits = {}
for split in ("train", "val", "test"):
    with (data_dir / "splits" / f"{split}.jsonl").open(encoding="utf-8") as f:
        splits[split] = [json.loads(line) for line in f if line.strip()]
    counts = Counter(obj["label"] for row in splits[split] for obj in row["objects"])
    print(f"{split}: {len(splits[split])} images, {sum(counts.values())} boxes | {dict(counts)}")

# Moi phien quay chi duoc xuat hien trong mot tap du lieu.
sessions = {name: {row["session"] for row in rows} for name, rows in splits.items()}
for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
    overlap = sessions[a] & sessions[b]
    assert not overlap, f"Overlapping sessions in {a}/{b}: {sorted(overlap)}"
print("OK: train/val/test have no overlapping sessions.")

train_records = splits["train"]
print("\nFirst 3 train records:")
for row in train_records[:3]:
    print(json.dumps(row, ensure_ascii=False, indent=2))

# Chon 6 anh co nhan; seed giup lan chay sau chon lai cung cac anh.
labeled_records = [row for row in train_records if row["objects"]]
if not labeled_records:
    raise ValueError("No labeled images found in train.jsonl")
samples = random.Random(42).sample(labeled_records, min(6, len(labeled_records)))
colors = {"pothole": "red", "crack": "yellow", "manhole": "cyan"}
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax in axes.flat:
    ax.axis("off")

for ax, row in zip(axes.flat, samples):
    with Image.open(data_dir / row["image_path"]) as image:
        image = image.convert("RGB")
    width, height = image.size
    ax.imshow(image)
    for obj in row["objects"]:
        # YOLO: tam (xc, yc), rong w, cao h deu nam tren thang 0-1.
        # Doi sang toa do pixel de ve khung len anh.
        xc, yc, w, h = obj["yolo"]
        x = (xc - w / 2) * width
        y = (yc - h / 2) * height
        box_width, box_height = w * width, h * height
        color = colors.get(obj["label"], "lime")
        ax.add_patch(Rectangle((x, y), box_width, box_height,
                               fill=False, edgecolor=color, linewidth=2))
        ax.text(x, y, obj["label"], color=color, fontsize=10,
                va="bottom", bbox=dict(facecolor="black", alpha=0.6, edgecolor="none"))
    ax.set_title(f"{row['image_id']} | {len(row['objects'])} boxes", fontsize=9)

plt.tight_layout()
plt.show()


## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
import json


def yolo_to_box(label, yolo):
    """Convert YOLO center/size (0-1) to corner coordinates (0-1000)."""
    xc, yc, w, h = yolo
    corners = [xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2]
    bbox = [max(0, min(1000, round(value * 1000))) for value in corners]
    return {"label": label, "bbox_2d": bbox}


def objects_to_answer(objects):
    """Return compact JSON, sorted from top to bottom, then left to right."""
    boxes = [yolo_to_box(obj["label"], obj["yolo"]) for obj in objects]
    boxes.sort(key=lambda box: (box["bbox_2d"][1], box["bbox_2d"][0]))
    return json.dumps(boxes, separators=(",", ":"))


PROMPT = (
    "Locate road damage in this image. "
    "Detect only objects of classes: pothole, crack, manhole. "
    "Output a JSON list of objects with label and bbox_2d coordinates "
    "normalized on a 0-1000 scale: "
    '[{"label": "...", "bbox_2d": [x1, y1, x2, y2]}]. '
    "If no damage is found, output []."
)

# Run the data cell above first: it defines train_records, data_dir and plotting imports.
example = next(row for row in train_records if row["objects"])
answer = objects_to_answer(example["objects"])
print("Image:", example["image_id"])
print("Answer:", answer)

# Compare original YOLO boxes with boxes converted from JSON back to pixels.
with Image.open(data_dir / example["image_path"]) as image:
    image = image.convert("RGB")
width, height = image.size
original_boxes = []
for obj in example["objects"]:
    xc, yc, w, h = obj["yolo"]
    original_boxes.append({
        "label": obj["label"],
        "bbox_2d": [(xc - w / 2) * width, (yc - h / 2) * height,
                    (xc + w / 2) * width, (yc + h / 2) * height],
    })
restored_boxes = [
    {"label": box["label"],
     "bbox_2d": [value / 1000 * size for value, size in
                 zip(box["bbox_2d"], (width, height, width, height))]}
    for box in json.loads(answer)
]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, boxes, title in zip(axes, (original_boxes, restored_boxes),
                            ("Original YOLO", "JSON 0-1000 -> pixels")):
    ax.imshow(image)
    for box in boxes:
        x1, y1, x2, y2 = box["bbox_2d"]
        color = colors.get(box["label"], "lime")
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1,
                               fill=False, edgecolor=color, linewidth=2))
        ax.text(x1, y1, box["label"], color=color, va="bottom",
                bbox=dict(facecolor="black", alpha=0.6, edgecolor="none"))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()


## 3. E0: zero-shot

In [ ]:
import gc
import json

import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration

if not torch.cuda.is_available():
    raise RuntimeError("Select a CUDA GPU (T4).")

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
BATCH_SIZE = 4


def load_fresh_base(model_id):
    quantization = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
        llm_int8_skip_modules=["model.visual", "visual", "lm_head"],
    )
    base = Qwen3VLForConditionalGeneration.from_pretrained(
        model_id, quantization_config=quantization,
        dtype=torch.float16, device_map={"": 0}, attn_implementation="sdpa",
    )
    from bitsandbytes.nn import Linear4bit, Linear8bitLt
    quantized = [name for name, module in base.named_modules()
                 if isinstance(module, (Linear4bit, Linear8bitLt))]
    if len(quantized) != 252 or any("language_model." not in name for name in quantized):
        raise RuntimeError("Expected 252 quantized LLM modules; vision and lm_head must stay floating point.")
    return base


def _predict_batch(model, rows):
    conversations = []
    for row in rows:
        with Image.open(data_dir / row["image_path"]) as image:
            rgb_image = image.convert("RGB")
        conversations.append([{"role": "user", "content": [
            {"type": "image", "image": rgb_image},
            {"type": "text", "text": PROMPT},
        ]}])
    inputs = processor.apply_chat_template(
        conversations, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt", padding=True,
    ).to(model.device)
    with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.float16):
        generated = model.generate(
            **inputs, max_new_tokens=256, do_sample=False,
            pad_token_id=processor.tokenizer.pad_token_id,
        )
    answer_ids = generated[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(
        answer_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False,
    )


def predict_records(model, records, output_path, batch_size=2, description="Inference"):
    if not records:
        raise ValueError("The inference split is empty.")
    if batch_size < 1:
        raise ValueError("batch_size must be positive.")
    model.eval()
    model.gradient_checkpointing_disable()
    model.config.use_cache = True
    processor.tokenizer.padding_side = "left"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    preview = []
    # L?u output th?; ch? gi?m batch n?u h?t VRAM, kh?ng b? qua ?nh.
    with output_path.open("w", encoding="utf-8") as output_file, tqdm(
        total=len(records), desc=description,
    ) as progress:
        start = 0
        while start < len(records):
            batch = records[start:start + batch_size]
            raw_answers = None
            try:
                raw_answers = _predict_batch(model, batch)
            except torch.cuda.OutOfMemoryError:
                if batch_size == 1:
                    raise
                batch_size = max(1, batch_size // 2)
                tqdm.write(f"Inference OOM: retry with batch_size={batch_size}.")
            # Ra kh?i except tr??c khi d?n b? nh? ?? gi?i ph?ng tensor trong traceback.
            if raw_answers is None:
                gc.collect()
                torch.cuda.empty_cache()
                continue
            for row, raw in zip(batch, raw_answers):
                prediction = {"image_id": row["image_id"], "raw": raw}
                output_file.write(json.dumps(prediction, ensure_ascii=False) + "\n")
                if len(preview) < 5:
                    preview.append(prediction)
            output_file.flush()
            start += len(batch)
            progress.update(len(batch))
    print(f"Saved {len(records)} predictions to {output_path}")
    for prediction in preview:
        print(prediction["image_id"], prediction["raw"], sep="\n")


processor = AutoProcessor.from_pretrained(MODEL_ID)
model = load_fresh_base(MODEL_ID)
try:
    predict_records(model, splits["test"], lab_dir / "work/e0.jsonl", BATCH_SIZE, "E0 test")
finally:
    del model
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable, str((lab_dir / "scripts/score.py").resolve()),
    "--pred", str((lab_dir / "work/e0.jsonl").resolve()), "--name", "E0",
    "--out", str((lab_dir / "report/evidence/e0.json").resolve()),
    "--splits-dir", str((data_dir / "splits").resolve()), "--split", "test",
], cwd=str(lab_dir.resolve()), check=True)

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
import torch
from torch.nn.utils.rnn import pad_sequence


def encode_example(image, prompt, answer):
    """Encode image/chat; supervise only the answer and <|im_end|>."""
    user = {
        "role": "user",
        "content": [
            {"type": "image", "image": image.convert("RGB")},
            {"type": "text", "text": prompt},
        ],
    }
    options = dict(tokenize=True, return_dict=True, return_tensors="pt")
    prompt_enc = processor.apply_chat_template(
        [user], add_generation_prompt=True, **options,
    )
    full_enc = processor.apply_chat_template(
        [user, {"role": "assistant", "content": answer}],
        add_generation_prompt=False, **options,
    )

    n = prompt_enc["input_ids"].shape[1]
    input_ids = full_enc["input_ids"]
    assert torch.equal(input_ids[:, :n], prompt_enc["input_ids"]), (
        "Prompt is not a prefix of the full conversation; check the chat template."
    )

    labels = input_ids.clone()
    labels[:, :n] = -100  # Ignore prompt, image tokens and assistant header.

    # Ignore any newline after the answer-ending token.
    end_id = processor.tokenizer.convert_tokens_to_ids("<|im_end|>")
    end_positions = (input_ids[0, n:] == end_id).nonzero(as_tuple=True)[0]
    assert end_positions.numel() > 0, "Answer is missing <|im_end|>."
    end = n + end_positions[0].item()
    labels[:, end + 1:] = -100
    full_enc["labels"] = labels
    return full_enc


def collate(batch):
    """Pad token sequences and concatenate image data into a batch."""
    if not batch:
        raise ValueError("Batch must not be empty.")
    pad_id = processor.tokenizer.pad_token_id
    if pad_id is None:
        raise ValueError("Tokenizer has no pad_token_id.")

    # Right-pad training batches independently of E0 padding_side="left".
    padding_values = {
        "input_ids": pad_id,
        "attention_mask": 0,
        "labels": -100,
    }
    if any("mm_token_type_ids" in example for example in batch):
        if not all("mm_token_type_ids" in example for example in batch):
            raise ValueError("All examples must provide mm_token_type_ids.")
        padding_values["mm_token_type_ids"] = 0

    result = {
        key: pad_sequence(
            [example[key].squeeze(0) for example in batch],
            batch_first=True,
            padding_value=value,
        )
        for key, value in padding_values.items()
    }
    for key in ("pixel_values", "image_grid_thw"):
        result[key] = torch.cat([example[key] for example in batch], dim=0)
    return result


# Check one training image using the variables from steps 1-3.
example = next(row for row in train_records if row["objects"])
answer = objects_to_answer(example["objects"])
with Image.open(data_dir / example["image_path"]) as image:
    image = image.convert("RGB")
encoded = encode_example(image, PROMPT, answer)

image_token_id = processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")
num_image_tokens = (encoded["input_ids"] == image_token_id).sum().item()
print(f"Image size: {image.size} | Image tokens: {num_image_tokens}")
if image.size == (640, 360):
    assert num_image_tokens == 220, f"Expected 220 image tokens, got {num_image_tokens}."

supervised_ids = encoded["labels"][encoded["labels"] != -100]
decoded_answer = processor.tokenizer.decode(
    supervised_ids.tolist(), skip_special_tokens=False,
    clean_up_tokenization_spaces=False,
)
print("Tokens used for loss:", decoded_answer)
assert decoded_answer == answer + "<|im_end|>", (
    "Supervised tokens must decode to the JSON answer plus <|im_end|>."
)

# Check padding and image concatenation with two different answer lengths.
empty_encoded = encode_example(image, PROMPT, "[]")
check_batch = collate([encoded, empty_encoded])
for key in ("attention_mask", "labels", "mm_token_type_ids"):
    if key in check_batch:
        assert check_batch[key].shape == check_batch["input_ids"].shape
assert torch.all(check_batch["labels"][check_batch["attention_mask"] == 0] == -100)
for key in ("pixel_values", "image_grid_thw"):
    assert check_batch[key].shape[0] == encoded[key].shape[0] + empty_encoded[key].shape[0]
print("OK: prompt mask, answer, padding and image concatenation.")
del check_batch, empty_encoded


## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
import re

import torch
from peft import LoraConfig, PeftModel, TaskType, get_peft_model


def apply_setup(model, setup):
    """Freeze base weights and configure trainable weights for E1, E2 or E3."""
    setup = setup.strip().upper()
    if setup not in {"E1", "E2", "E3"}:
        raise ValueError("setup must be E1, E2 or E3.")
    if isinstance(model, PeftModel) or getattr(model, "peft_config", None):
        raise ValueError("Load a fresh base model for each experiment before apply_setup.")

    llm_pattern = (
        r"model\.language_model\.layers\.\d+\."
        r"(?:self_attn\.(?:q|k|v|o)_proj|mlp\.(?:gate|up|down)_proj)"
    )
    vision_pattern = (
        r"model\.visual\.blocks\.\d+\."
        r"(?:attn\.(?:qkv|proj)|mlp\.(?:linear_fc1|linear_fc2))"
    )
    target_pattern = llm_pattern
    if setup == "E3":
        target_pattern = rf"(?:{llm_pattern}|{vision_pattern})"

    matched_modules = [
        name for name, _ in model.named_modules()
        if re.fullmatch(target_pattern, name)
    ]
    expected = 348 if setup == "E3" else 252
    if len(matched_modules) != expected:
        raise ValueError(
            f"{setup}: matched {len(matched_modules)} LoRA modules, expected {expected}. "
            "Check the model architecture and module names."
        )

    # The full path is required to exclude vision from 4-bit conversion.
    from bitsandbytes.nn import Linear4bit, Linear8bitLt
    quantized_visual = [
        name for name, module in model.named_modules()
        if name.startswith("model.visual.")
        and isinstance(module, (Linear4bit, Linear8bitLt))
    ]
    if quantized_visual:
        raise RuntimeError(
            f"Vision must not be quantized: {quantized_visual[:5]}. "
            'Reload with llm_int8_skip_modules=["model.visual", "lm_head"].'
        )

    for parameter in model.parameters():
        parameter.requires_grad = False
    # Keep gradients flowing through frozen embeddings during checkpointing.
    model.enable_input_require_grads()

    config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=target_pattern,
        bias="none",
    )
    peft_model = get_peft_model(model, config)

    if setup in {"E2", "E3"}:
        for name, parameter in peft_model.named_parameters():
            if "visual.merger" in name or "visual.deepstack_merger_list" in name:
                parameter.data = parameter.data.to(dtype=torch.float32)
                parameter.requires_grad = True

    # LoRA v? projector c?n FP32 ?? GradScaler unscale gradient an to?n.
    for name, parameter in peft_model.named_parameters():
        if parameter.requires_grad:
            if "lora_" not in name and not (
                setup in {"E2", "E3"} and
                ("visual.merger." in name or "visual.deepstack_merger_list." in name)
            ):
                raise RuntimeError(f"Unexpected trainable parameter: {name}")
            parameter.data = parameter.data.to(dtype=torch.float32)
    print(f"{setup}: {len(matched_modules)} LoRA target modules")
    peft_model.print_trainable_parameters()
    return peft_model


# Run with a freshly loaded base model for each experiment:
# model = apply_setup(model, "E1")  # LLM LoRA only.
# model = apply_setup(model, "E2")  # LLM LoRA + train all merger weights.
# model = apply_setup(model, "E3")  # LLM/vision LoRA + train all merger weights.

## 6. Vòng huấn luyện

In [ ]:
import gc
import math
import random
import time

import torch
from tqdm.auto import tqdm
from transformers import AutoProcessor

MAX_STEPS = 300
TRAIN_BATCH_SIZE = 2
GRAD_ACCUM = 4  
LR = 2e-4
SEED = 42
SAVE_EVERY = 100

if not torch.cuda.is_available():
    raise RuntimeError("Training requires a CUDA GPU (T4).")
if not train_records:
    raise ValueError("The training split is empty.")

for variable in ("scheduler", "optimizer", "params", "scaler", "model"):
    globals().pop(variable, None)
gc.collect()
torch.cuda.empty_cache()
processor = AutoProcessor.from_pretrained(MODEL_ID)


def _train_once(setup, records, run_name, batch_size, grad_accum):
    run_dir = lab_dir / "work" / (run_name or setup.lower())
    random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    model = optimizer = scheduler = scaler = params = batch = loss = None
    try:
        model = apply_setup(load_fresh_base(MODEL_ID), setup)
        trainable_names = {name for name, p in model.named_parameters() if p.requires_grad}
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.config.use_cache = False
        for name, parameter in model.named_parameters():
            parameter.requires_grad_(name in trainable_names)
            parameter.grad = None
            if parameter.requires_grad:
                parameter.data = parameter.data.to(dtype=torch.float32)
        params = [p for p in model.parameters() if p.requires_grad]
        if not params or any(p.dtype != torch.float32 for p in params):
            raise ValueError("Trainable weights must be nonempty and float32.")
        trainable_params = sum(p.numel() for p in params)
        
        optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=0.0, foreach=False)
        warmup_steps = max(1, math.ceil(MAX_STEPS * 0.05))

        def lr_multiplier(step):
            if step < warmup_steps:
                return step / warmup_steps
            progress = (step - warmup_steps) / max(1, MAX_STEPS - warmup_steps)
            return 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))

        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_multiplier)
        scaler = torch.amp.GradScaler("cuda", init_scale=256.0)
        model.train()
        optimizer.zero_grad(set_to_none=True)
        run_dir.mkdir(parents=True, exist_ok=True)
        torch.cuda.synchronize(model.device)
        torch.cuda.reset_peak_memory_stats(model.device)
        started = time.perf_counter()
        loss_history = []
        overflow_skips = 0

        def save_checkpoint(destination, completed_steps):
            destination.mkdir(parents=True, exist_ok=True)
            model.save_pretrained(destination)
            processor.save_pretrained(destination)
            
            if setup in {"E2", "E3"}:
                extra = {name: p.detach().float().cpu().clone()
                         for name, p in model.named_parameters()
                         if p.requires_grad and
                         ("visual.merger." in name or "visual.deepstack_merger_list." in name)}
                if not extra:
                    raise RuntimeError("Missing trainable projector weights.")
                torch.save(extra, destination / "extra_trainable.pt")
            torch.cuda.synchronize(model.device)
            meta = {
                "setup": setup, "model_id": MODEL_ID, "steps": completed_steps,
                "n_train_images": len(records), "trainable_params": trainable_params,
                "peak_vram_gb": torch.cuda.max_memory_allocated(model.device) / 1024**3,
                "train_minutes": (time.perf_counter() - started) / 60,
                "batch_size": batch_size, "grad_accum": grad_accum,
                "lr": LR, "seed": SEED, "amp_overflow_skips": overflow_skips,
                "loss_history": loss_history,
            }
            (destination / "run.json").write_text(
                json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8",
            )
            return meta

        rng = random.Random(SEED)
        order = list(range(len(records)))
        cursor = len(order)
        step = consecutive_overflows = 0
        with tqdm(total=MAX_STEPS, desc=f"Train {setup}") as progress_bar:
            while step < MAX_STEPS:
                step_loss = 0.0
                for micro_step in range(grad_accum):
                    examples = []
                    
                    for _ in range(batch_size):
                        if cursor >= len(order):
                            rng.shuffle(order)
                            cursor = 0
                        row = records[order[cursor]]
                        cursor += 1
                        with Image.open(data_dir / row["image_path"]) as image:
                            examples.append(encode_example(image, PROMPT, objects_to_answer(row["objects"])))
                    batch = {key: value.to(model.device) for key, value in collate(examples).items()}
                    del examples
                    with torch.autocast(device_type="cuda", dtype=torch.float16):
                        loss = model(**batch).loss / grad_accum
                    if not torch.isfinite(loss).item():
                        raise FloatingPointError(f"Non-finite loss at step {step + 1}, micro-batch {micro_step + 1}.")
                    step_loss += loss.detach().item()
                    scaler.scale(loss).backward()
                    batch = loss = None

                invalid_grads = [name for name, p in model.named_parameters()
                                 if p.grad is not None and
                                 (not p.requires_grad or p.grad.dtype != torch.float32)]
                if invalid_grads:
                    raise RuntimeError(f"Unexpected gradients: {invalid_grads[:10]}")
                scaler.unscale_(optimizer)
                previous_scale = scaler.get_scale()
                grad_norm = torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                if scaler.get_scale() < previous_scale:
                    overflow_skips += 1
                    consecutive_overflows += 1
                    if consecutive_overflows >= 20:
                        raise FloatingPointError("20 consecutive gradient overflows; check fp16 stability.")
                    tqdm.write(f"Gradient overflow: scale={scaler.get_scale()}, successful steps={step}.")
                    continue
                if not torch.isfinite(grad_norm).item():
                    raise FloatingPointError("Non-finite gradient norm.")
                consecutive_overflows = 0
                step += 1
                scheduler.step()
                progress_bar.update(1)
                loss_history.append({"step": step, "loss": step_loss,
                                     "grad_norm": grad_norm.item(), "lr": optimizer.param_groups[0]["lr"]})
                if step == 1 or step % 10 == 0:
                    tqdm.write(f"step={step}/{MAX_STEPS} loss={step_loss:.4f} "
                               f"lr={optimizer.param_groups[0]['lr']:.2e}")
                if step % SAVE_EVERY == 0:
                    save_checkpoint(run_dir / f"checkpoint-{step}", step)
        run_meta = save_checkpoint(run_dir, MAX_STEPS)
        print(f"Saved adapter and run.json to {run_dir}")
        print(json.dumps({key: run_meta[key] for key in
                          ("trainable_params", "peak_vram_gb", "train_minutes")}, indent=2))
        return run_meta
    finally:
        del model, optimizer, scheduler, scaler, params, batch, loss
        gc.collect()
        torch.cuda.empty_cache()


def train_setup(setup, records=None, run_name=None, batch_size=None, grad_accum=None, allow_fallback=True):
    records = train_records if records is None else records
    batch_size = TRAIN_BATCH_SIZE if batch_size is None else batch_size
    grad_accum = GRAD_ACCUM if grad_accum is None else grad_accum
    if not records or batch_size < 1 or grad_accum < 1:
        raise ValueError("Training records and batch settings must be nonempty and positive.")
    try:
        return _train_once(setup, records, run_name, batch_size, grad_accum)
    except torch.cuda.OutOfMemoryError:
        if not allow_fallback or batch_size == 1:
            raise
        print("Training OOM: restart from the base model with batch_size=1.")
    gc.collect()
    torch.cuda.empty_cache()
    return _train_once(setup, records, run_name, 1, batch_size * grad_accum)


training_runs = globals().get("training_runs", {})


In [ ]:
# Huấn luyện E1
print("=== Training E1 ===")
training_runs["E1"] = train_setup("E1")
gc.collect()
torch.cuda.empty_cache()
print("Finished training E1.")


In [ ]:
# Huấn luyện E2
print("=== Training E2 ===")
training_runs["E2"] = train_setup("E2")
gc.collect()
torch.cuda.empty_cache()
print("Finished training E2.")


In [ ]:
# Huấn luyện E3
print("=== Training E3 ===")
training_runs["E3"] = train_setup("E3")
gc.collect()
torch.cuda.empty_cache()
print("Finished training E3.")


## 7. Suy luận và chấm E1–E3

In [ ]:
import gc
import json
import math
import random
import subprocess
import sys

import torch
from peft import PeftModel
from transformers import AutoProcessor
from scripts.score import SCORE_VERSION, evaluate

EVAL_SETUPS = ("E1", "E2", "E3")
INFER_BATCH_SIZE = 2

validation_records = random.Random(SEED).sample(splits["val"], min(40, len(splits["val"])))
if not validation_records:
    raise ValueError("The validation split is empty.")


def evaluate_setup(setup, run_name=None, score_name=None, validate=False):
    global processor
    run_name = run_name or setup.lower()
    adapter_dir = (lab_dir / "work" / run_name).resolve()
    output_path = (lab_dir / "work" / f"{run_name}.jsonl").resolve()
    evidence_path = (lab_dir / "report/evidence" / f"{run_name}.json").resolve()
    if setup not in {"E1", "E2", "E3"}:
        raise ValueError("setup must be E1, E2 or E3.")
    for filename in ("adapter_config.json", "run.json"):
        if not (adapter_dir / filename).is_file():
            raise FileNotFoundError(f"Missing {adapter_dir / filename}; finish step 6 first.")
    run_meta = json.loads((adapter_dir / "run.json").read_text(encoding="utf-8"))
    if run_meta.get("setup") != setup or run_meta.get("steps") != MAX_STEPS:
        raise ValueError("Saved run.json does not match the setup or completed training steps.")
    meta_keys = ("trainable_params", "peak_vram_gb", "train_minutes")
    for key in meta_keys:
        value = run_meta.get(key)
        if (isinstance(value, bool) or not isinstance(value, (int, float))
                or not math.isfinite(value) or value <= 0):
            raise ValueError(f"Invalid {key} in run.json: {value!r}")
    projector_path = adapter_dir / "extra_trainable.pt"
    if setup in {"E2", "E3"} and not projector_path.is_file():
        raise FileNotFoundError(f"Missing trained projector: {projector_path}")

    model = None
    try:
        model = load_fresh_base(run_meta["model_id"])
        model = PeftModel.from_pretrained(model, adapter_dir, is_trainable=False)
        if setup in {"E2", "E3"}:
            extra = torch.load(projector_path, map_location="cpu", weights_only=True)
            named = {name: p for name, p in model.named_parameters()
                     if "visual.merger." in name or "visual.deepstack_merger_list." in name}
            if not named or set(extra) != set(named):
                raise ValueError("Saved projector keys do not match the reloaded model.")
            with torch.no_grad():
                for name, parameter in named.items():
                    value = extra[name]
                    if value.shape != parameter.shape or value.dtype != torch.float32:
                        raise ValueError(f"Invalid projector tensor: {name}")
                    parameter.data = parameter.data.float()
                    parameter.copy_(value.to(parameter.device))
                    if not torch.equal(parameter.detach().cpu(), value):
                        raise ValueError(f"Projector reload mismatch: {name}")
            del extra, named, parameter, value
        processor = AutoProcessor.from_pretrained(adapter_dir)
        if validate:
            val_prediction_path = lab_dir / "work" / f"{run_name}_val.jsonl"
            predict_records(model, validation_records, val_prediction_path, INFER_BATCH_SIZE, f"{setup} val")
            predictions = [json.loads(line) for line in val_prediction_path.read_text(encoding="utf-8").splitlines()]
            val_score = evaluate(validation_records, {row["image_id"]: row["raw"] for row in predictions})
            dataset_meta = json.loads((data_dir / "splits/meta.json").read_text(encoding="utf-8"))
            val_score.update({"score_version": SCORE_VERSION, "name": setup,
                              "dataset_version": dataset_meta["dataset_version"],
                              "split": "val", "coord": "norm1000", "meta": run_meta,
                              "image_ids": [row["image_id"] for row in validation_records]})
            val_path = lab_dir / "report/validation" / f"{run_name}.json"
            val_path.parent.mkdir(parents=True, exist_ok=True)
            val_path.write_text(json.dumps(val_score, ensure_ascii=False, indent=2), encoding="utf-8")
        predict_records(model, splits["test"], output_path, INFER_BATCH_SIZE, f"{setup} test")
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()

    command = [sys.executable, str((lab_dir / "scripts/score.py").resolve()),
               "--pred", str(output_path), "--name", score_name or setup, "--out", str(evidence_path),
               "--splits-dir", str((data_dir / "splits").resolve()), "--split", "test"]
    for key in meta_keys:
        value = run_meta[key]
        formatted = str(value) if isinstance(value, int) else f"{value:.16f}"
        command.extend(["--meta", f"{key}={formatted}"])
    subprocess.run(command, cwd=str(lab_dir.resolve()), check=True)
    return {"predictions": str(output_path), "scores": str(evidence_path)}


evaluation_files = {}
for setup in EVAL_SETUPS:
    print(f"\n=== Evaluating {setup} ===")
    evaluation_files[setup] = evaluate_setup(setup, validate=True)
    gc.collect()
    torch.cuda.empty_cache()
for setup, files in evaluation_files.items():
    print(f"{setup}: {files['predictions']}")
    print(f"{setup}: {files['scores']}")

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# GUIDE bước 8: chỉ đổi data_size, không thực hiện bước 9.
import copy
import gc
import json
import math
import random
import shutil

# Chạy các cell bước 1-7 trước. Dùng F1 đã chấm để chọn setup gốc.
evidence_dir = lab_dir / "report" / "evidence"
baseline_scores = {}
validation_scores = {}
for setup in ("E1", "E2", "E3"):
    path = evidence_dir / f"{setup.lower()}.json"
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}; finish step 7 first.")
    score = json.loads(path.read_text(encoding="utf-8"))
    if (score.get("split") != "test" or score.get("coord") != "norm1000"
            or score.get("n_images") != len(splits["test"])
            or score.get("missing_predictions") != 0):
        raise ValueError(f"{setup}: expected complete test scores on norm1000 coordinates.")
    f1 = score["micro"]["f1"]
    if not isinstance(f1, (int, float)) or not math.isfinite(f1):
        raise ValueError(f"{setup}: invalid F1.")
    baseline_scores[setup] = score
    val_path = lab_dir / "report/validation" / f"{setup.lower()}.json"
    val_score = json.loads(val_path.read_text(encoding="utf-8"))
    expected_ids = [row["image_id"] for row in validation_records]
    if (val_score.get("split") != "val" or val_score.get("coord") != "norm1000"
            or val_score.get("image_ids") != expected_ids
            or val_score.get("dataset_version") != score.get("dataset_version")
            or val_score.get("missing_predictions") != 0):
        raise ValueError(f"{setup}: invalid or inconsistent validation scores.")
    val_f1 = val_score["micro"]["f1"]
    if not isinstance(val_f1, (int, float)) or not math.isfinite(val_f1):
        raise ValueError(f"{setup}: invalid validation F1.")
    validation_scores[setup] = val_score

# Nếu F1 bằng nhau, ưu tiên E1 rồi E2 rồi E3.
best_setup = max(validation_scores, key=lambda setup: validation_scores[setup]["micro"]["f1"])
baseline_dir = lab_dir / "work" / best_setup.lower()
baseline_meta = json.loads((baseline_dir / "run.json").read_text(encoding="utf-8"))
# Bảo đảm các thiết lập hiện tại khớp lần train 100% đã lưu.
expected = {"setup": best_setup, "model_id": MODEL_ID, "steps": MAX_STEPS,
            "lr": LR, "seed": SEED}
for key, value in expected.items():
    if baseline_meta.get(key) != value:
        raise ValueError(f"Restore {key}={baseline_meta.get(key)!r} from the baseline before E4.")
if baseline_meta.get("n_train_images", len(train_records)) != len(train_records):
    raise ValueError("The baseline was not trained on the current full training split.")
baseline_batch_size = baseline_meta["batch_size"]
baseline_grad_accum = baseline_meta["grad_accum"]
if baseline_batch_size * baseline_grad_accum != TRAIN_BATCH_SIZE * GRAD_ACCUM:
    raise ValueError("The baseline effective batch size differs from the current configuration.")
baseline_predictions = lab_dir / "work" / f"{best_setup.lower()}.jsonl"
if not baseline_predictions.is_file():
    raise FileNotFoundError(baseline_predictions)

# Lấy ngẫu nhiên 10% số ảnh, gồm cả ảnh không có đối tượng; không lấy từ val/test.
subset_size = max(1, math.ceil(len(train_records) * 0.1))
indices = sorted(random.Random(SEED).sample(range(len(train_records)), subset_size))
subset_records = [train_records[index] for index in indices]
run_name = "e4_data_size_0.1"
subset_dir = lab_dir / "work" / run_name
subset_dir.mkdir(parents=True, exist_ok=True)
(subset_dir / "train_subset.jsonl").write_text(
    "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in subset_records),
    encoding="utf-8",
)
print(f"Best setup: {best_setup}, validation F1={validation_scores[best_setup]['micro']['f1']:.4f}")
print(f"Train images: 10%={subset_size}, 100%={len(train_records)}; seed={SEED}")
print(f"Both use {MAX_STEPS} optimizer steps and identical model, LoRA, LR, batch and prompt.")

# 100% chính là setup gốc: dùng lại kết quả, không train/chấm test lần nữa.
full_name = "e4_data_size_1.0"
full_prediction_path = lab_dir / "work" / f"{full_name}.jsonl"
shutil.copyfile(baseline_predictions, full_prediction_path)
full_score = copy.deepcopy(baseline_scores[best_setup])
full_score["name"] = "E4_data_size_1.0"
full_score["meta"].update({"base_setup": best_setup, "ablation_variable": "data_size",
                           "train_fraction": 1.0, "n_train_images": len(train_records),
                           "seed": SEED, "source_run": str(baseline_dir), "selection_split": "val",
                           "batch_size": baseline_batch_size, "grad_accum": baseline_grad_accum})
full_evidence_path = evidence_dir / f"{full_name}.json"
full_evidence_path.write_text(json.dumps(full_score, ensure_ascii=False, indent=2), encoding="utf-8")

# Train 10% từ model gốc mới, không tiếp tục adapter của bản 100%.
for variable in ("scheduler", "optimizer", "params", "scaler", "model"):
    globals().pop(variable, None)
gc.collect()
torch.cuda.empty_cache()
small_meta = train_setup(
    best_setup, records=subset_records, run_name=run_name,
    batch_size=baseline_batch_size, grad_accum=baseline_grad_accum, allow_fallback=False,
)
small_meta.update({"base_setup": best_setup, "ablation_variable": "data_size",
                   "train_fraction": 0.1})
(subset_dir / "run.json").write_text(
    json.dumps(small_meta, ensure_ascii=False, indent=2), encoding="utf-8",
)
gc.collect()
torch.cuda.empty_cache()
small_files = evaluate_setup(best_setup, run_name=run_name, score_name="E4_data_size_0.1")
gc.collect()
torch.cuda.empty_cache()
small_evidence_path = Path(small_files["scores"])
small_score = json.loads(small_evidence_path.read_text(encoding="utf-8"))
small_score["meta"].update({"base_setup": best_setup, "ablation_variable": "data_size",
                            "train_fraction": 0.1, "n_train_images": subset_size, "seed": SEED, "selection_split": "val",
                            "batch_size": baseline_batch_size, "grad_accum": baseline_grad_accum})
small_evidence_path.write_text(
    json.dumps(small_score, ensure_ascii=False, indent=2), encoding="utf-8",
)

e4_results = {"0.1": small_score, "1.0": full_score}
print("\nData size | Images | F1     | CI95          | FP/image | VRAM GB | Minutes")
for fraction, score in e4_results.items():
    lo, hi = score["f1_ci95"]
    meta = score["meta"]
    print(f"{float(fraction):>8.0%} | {meta['n_train_images']:>6} | "
          f"{score['micro']['f1']:.4f} | [{lo:.4f}, {hi:.4f}] | "
          f"{score['fp_per_image']:.4f}   | {meta['peak_vram_gb']:.2f}    | "
          f"{meta['train_minutes']:.2f}")
delta = full_score["micro"]["f1"] - small_score["micro"]["f1"]
print(f"\nF1(100%) - F1(10%) = {delta:+.4f}")
small_lo, small_hi = small_score["f1_ci95"]
full_lo, full_hi = full_score["f1_ci95"]
if max(small_lo, full_lo) <= min(small_hi, full_hi):
    print("CI95 chồng lấn: chưa đủ cơ sở để khẳng định chắc chắn bản nào tốt hơn.")
else:
    print("CI95 không chồng lấn: kết quả gợi ý có khác biệt, nhưng chưa phải kiểm định chênh lệch F1.")
print(f"Test có {full_score['n_images']} ảnh; kết luận chỉ áp dụng cho tập này và seed đã dùng.")
print("Setup gốc được chọn theo F1 test đã có, nên so sánh này mang tính thăm dò.")
print("Evidence:", small_evidence_path, full_evidence_path, sep="\n")